# Automated Pelvimetry Analysis Demo

This notebook demonstrates how to use the `pelvimetry_core` library to extract anatomical metrics from a CT scan.

**Note**: This demo uses the sample data provided in the `Demo` folder. It bypasses the time-consuming `TotalSegmentator` step because segmentation masks are already provided.

In [ ]:
import os
import pandas as pd
from pelvimetry_core import AutomatedPelvimetry

## 1. Setup Data Paths
We point to the local `Demo` folder which contains `Patient_CT.nii.gz` and pre-computed masks.

In [ ]:
# Path to the sample NIfTI file
NIFTI_FILE = "./Demo/Patient_CT.nii.gz"

# Directory containing the segmentation masks (hip, sacrum, etc.)
# Since masks are already in ./Demo, we point output_dir there.
OUTPUT_DIR = "./Demo"

## 2. Initialize Pipeline

In [ ]:
pipeline = AutomatedPelvimetry()

if not os.path.exists(NIFTI_FILE):
    print(f"Warning: Demo file {NIFTI_FILE} not found.")
else:
    print(f"Found input: {NIFTI_FILE}")

## 3. Run Analysis

In [ ]:
# Manual Step-by-Step Execution for Demo Folder Structure

# 1. Load Data directly from Demo folder
print(f"Loading masks from {OUTPUT_DIR}...")
data, affine, spacing = pipeline.load_data(OUTPUT_DIR)

# 2. Calculate ISD
print("Calculating ISD...")
isd_res, trace = pipeline.calculate_isd(data, spacing)

# 3. Calculate APD (Anteroposterior Diameter)
print("Calculating APD...")
apd_res = pipeline.calculate_apd(data, isd_res, spacing)

# 4. Calculate Triangle Metrics (includes pPFA)
print("Calculating Triangle Metrics...")
tri_res = pipeline.calculate_triangle_metrics(data, isd_res, spacing)

# 5. Calculate IPFA
print("Calculating IPFA...")
ipfa_val, ipfa_status = pipeline.calculate_ipfa(data, isd_res, NIFTI_FILE, spacing)

# 6. Consolidate Results (Simulate pipeline_single_case flattening)
results = {}
results.update(isd_res)
results.update(apd_res)
results.update(tri_res)
results["IPFA_cm2"] = ipfa_val
results["IPFA_Status"] = ipfa_status

# Helper to flatten like pipeline_single_case does
if "pt_L" in isd_res and isd_res["pt_L"] is not None:
    results["ISD_L_x"] = int(isd_res["pt_L"][0])
    results["ISD_L_y"] = int(isd_res["pt_L"][1])
if "pt_R" in isd_res and isd_res["pt_R"] is not None:
    results["ISD_R_x"] = int(isd_res["pt_R"][0])
    results["ISD_R_y"] = int(isd_res["pt_R"][1])
if "Point_C_x" in tri_res:
    # These might be float, cast to valid type if needed
    results["Point_C_x"] = tri_res["Point_C_x"]
    results["Point_C_y"] = tri_res["Point_C_y"]

print("\n=== Analysis Results ===")
metrics = [
    "ISD_mm", "APD_mm", 
    "Triangle_Area_cm2", "Triangle_Depth_mm", "Triangle_Shape_Index",
    "pPFA_cm2", "Fat_Occupancy_Ratio",
    "Rectum_Area_Triangle_cm2", "Rectum_Occupancy_Ratio",
    "Operating_Space_cm2", 
    "IPFA_cm2",
    # Coordinates
    "ISD_L_x", "ISD_L_y",
    "ISD_R_x", "ISD_R_y",
    "Point_C_x", "Point_C_y"
]

for k in metrics:
    v = results.get(k)
    if v is None: 
        print(f"{k}: None")
    elif isinstance(v, float):
        print(f"{k}: {v:.2f}")
    else:
        print(f"{k}: {v}")

## 4. Expected Output for Demo Data
You should see:
*   **ISD_mm**: ~86.15 mm
*   **APD_mm**: ~120.25 mm
*   **Triangle Shape Index**: ~0.432
*   **pPFA_cm2**: ~13.26 cm²
*   **IPFA_cm2**: ~25.97 cm²
*   **Flattened Coordinates**: e.g. ISD_L_x, Point_C_x populated.